# Joins in SLayer

Joins connect models so that dimensions and measures from one model are accessible when querying another. SLayer only supports **LEFT JOINs** — the kind most commonly used for data enrichment.

If you think of left joins as directed edges of a graph whose vertices are models, then the dimensions, measures, and filters in a model have access to columns from any model reachable in the join graph.

This notebook illustrates every aspect of joins with working code.

See also: [Models — Joins](../../concepts/models.md#joins) | [Queries — Cross-Model Measures](../../concepts/queries.md#cross-model-measures) | [Ingestion — Diamond Joins](../../concepts/ingestion.md#diamond-joins)

**Prerequisites:** `pip install motley-slayer[examples]`

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

from slayer.core.query import SlayerQuery

engine, storage, models = ensure_jaffle_shop()

## Basic Joins

A join is defined by a **target model** and a list of **join pairs** — column pairs to join on. The `orders` model has two joins, auto-generated from its foreign keys to `customers` and `stores`:

In [ ]:
orders_model = next(m for m in models if m.name == "orders")

print("orders model joins:")
for j in orders_model.joins:
    print(f"  -> {j.target_model}  ON {j.join_pairs}")

# Query using a joined dimension
result = engine.execute(query=SlayerQuery(
    source_model="orders",
    fields=[{"formula": "count"}, {"formula": "order_total_sum"}],
    dimensions=[{"name": "customers.name"}],
    order=[{"column": {"name": "order_total_sum"}, "direction": "desc"}],
    limit=5,
))

print("\nTop 5 customers by revenue:")
for row in result.data:
    print(f"  {row['orders.customers.name']}: {row['orders.count']} orders, ${row['orders.order_total_sum']:,.2f}")

## Referencing Joined Models in Queries (Dot Syntax)

SLayer uses **dot syntax** to reference dimensions and measures from joined models:

- 1-hop: `customers.name` (orders -> customers)
- Multi-hop: `orders.customers.name` (order_items -> orders -> customers)

The full path avoids ambiguity when there are multiple ways to reach a model.

In [ ]:
# 1-hop: orders -> stores
result = engine.execute(query=SlayerQuery(
    source_model="orders",
    fields=[{"formula": "count"}],
    dimensions=[{"name": "stores.name"}],
))

print("1-hop (orders -> stores):")
for row in result.data:
    print(f"  {row['orders.stores.name']}: {row['orders.count']} orders")

In [ ]:
# Multi-hop: order_items -> orders -> customers
result = engine.execute(query=SlayerQuery(
    source_model="order_items",
    fields=[{"formula": "count"}, {"formula": "quantity_sum"}],
    dimensions=[{"name": "orders.customers.name"}],
    order=[{"column": {"name": "quantity_sum"}, "direction": "desc"}],
    limit=5,
))

print("Multi-hop (order_items -> orders -> customers):")
for row in result.data:
    print(f"  {row['order_items.orders.customers.name']}: {row['order_items.count']} line items, {row['order_items.quantity_sum']} qty")

## Referencing Joined Models in SQL Snippets

When defining dimensions and measures at the model level (in YAML or Python), their `sql` fields use **double underscores** instead of dots for joined table aliases. This is because dots aren't valid in SQL identifiers.

| Context | Syntax | Example |
|---------|--------|---------|
| Query dimensions/measures | dots | `orders.customers.name` |
| Model SQL expressions | `__` aliases | `orders__customers.name` |

SLayer substitutes the `__` aliases for correct table aliases at query time.

In [ ]:
oi_model = next(m for m in models if m.name == "order_items")

# Show the name vs sql for multi-hop dimensions
print(f"{'Dimension name (dots)':<30} {'SQL expression (__ alias)':<30}")
print("-" * 62)
for dim in oi_model.dimensions:
    if "__" in (dim.sql or ""):
        print(f"{dim.name:<30} {dim.sql:<30}")

## Auto-Ingesting Schemas

When auto-ingesting a database, SLayer introspects FK constraints and creates `ModelJoin` objects automatically. It follows FK chains transitively — if `order_items` -> `orders` -> `customers`, then the `order_items` model gets joins to all three tables.

Multi-hop join pairs use path-qualified source columns to indicate which already-joined table the FK comes from:

In [ ]:
import sqlalchemy as sa

from setup_jaffle import DB_PATH
from slayer.core.models import DatasourceConfig
from slayer.engine.ingestion import _build_fk_graph

ds = DatasourceConfig(name="jaffle_shop", type="duckdb", database=DB_PATH)
sa_engine = sa.create_engine(ds.resolve_env_vars().get_connection_string())
inspector = sa.inspect(sa_engine)
fk_graph = _build_fk_graph(inspector=inspector, table_names=inspector.get_table_names(), schema=None)
sa_engine.dispose()

print("FK graph:")
for table in sorted(fk_graph):
    print(f"  {table} -> {sorted(fk_graph[table])}")

print("\norder_items joins (auto-generated):")
for j in oi_model.joins:
    src = j.join_pairs[0][0]
    tgt = j.join_pairs[0][1]
    hop = "multi-hop" if "." in src else "direct"
    print(f"  -> {j.target_model:<12} ON {src:<20} = {tgt:<5} ({hop})")

## Diamond Joins

A **diamond join** occurs when the same table is reachable via multiple FK paths. For example:

```
orders -> customers -> regions
orders -> warehouses -> regions
```

SLayer treats each path as a separate copy of the target table, using **path-based aliases** to disambiguate:
- `customers.regions.name` -> table alias `customers__regions`  
- `warehouses.regions.name` -> table alias `warehouses__regions`

The Jaffle Shop schema doesn't have a natural diamond, so let's construct one:

In [ ]:
from joins_utils import setup_diamond_example

diamond_engine, diamond_storage, diamond_models, diamond_db_path, diamond_work_dir = setup_diamond_example()

diamond_orders = next(m for m in diamond_models if m.name == "orders")

print("Diamond orders model joins:")
for j in diamond_orders.joins:
    src = j.join_pairs[0][0]
    tgt = j.join_pairs[0][1]
    print(f"  -> {j.target_model:<12} ON {src:<25} = {tgt}")

print("\nDimensions with path aliases:")
for dim in diamond_orders.dimensions:
    if "regions" in dim.name:
        print(f"  name: {dim.name:<35} sql: {dim.sql}")

In [ ]:
# Query both paths simultaneously
result = diamond_engine.execute(query=SlayerQuery(
    source_model="orders",
    fields=[{"formula": "count"}, {"formula": "amount_sum"}],
    dimensions=[
        {"name": "customers.regions.name"},
        {"name": "warehouses.regions.name"},
    ],
))

print(f"{'Customer Region':<18} {'Warehouse Region':<18} {'Orders':>7} {'Amount':>10}")
print("-" * 56)
for row in result.data:
    cr = row["orders.customers.regions.name"]
    wr = row["orders.warehouses.regions.name"]
    print(f"{cr:<18} {wr:<18} {row['orders.count']:>7} ${row['orders.amount_sum']:>9,.2f}")

### Recombining Diamond Joins with Filters

By default, each path to the same table produces independent copies. If you want to enforce that the two paths resolve to the same row (re-creating a true diamond), add a query-level filter equating the two paths.

Query-level filters reference dimension names using dot syntax — SLayer resolves them to the correct SQL aliases automatically:

In [ ]:
# Only keep orders where customer and warehouse are in the same region
result = diamond_engine.execute(query=SlayerQuery(
    source_model="orders",
    fields=[{"formula": "count"}, {"formula": "amount_sum"}],
    dimensions=[{"name": "customers.regions.name"}],
    filters=["customers.regions.name == warehouses.regions.name"],
))

print("Orders where customer and warehouse are in the same region:")
for row in result.data:
    print(f"  {row['orders.customers.regions.name']}: {row['orders.count']} orders, ${row['orders.amount_sum']:,.2f}")

## Dynamic Joins (ModelExtension)

Joins can be added at query time via `ModelExtension`, without modifying the stored model. This is useful for:
- Ad-hoc joins to lookup tables
- Joining to models created dynamically from queries
- Adding context-specific enrichment

Let's create a customer segments table and join it to orders on the fly:

In [ ]:
import duckdb

from setup_jaffle import DB_PATH
from slayer.core.models import Dimension, Measure, SlayerModel

# Create a segments table in the existing DB
conn = duckdb.connect(DB_PATH)
conn.execute("""
    CREATE TABLE IF NOT EXISTS customer_segments AS
    SELECT
        c.id,
        CASE
            WHEN order_count >= 200 THEN 'vip'
            WHEN order_count >= 100 THEN 'regular'
            ELSE 'occasional'
        END AS segment
    FROM customers c
    JOIN (
        SELECT customer_id, COUNT(*) as order_count
        FROM orders GROUP BY customer_id
    ) o ON c.id = o.customer_id
""")
conn.close()

# Define a model for the segments table (without saving it)
segments_model = SlayerModel(
    name="customer_segments",
    sql_table="customer_segments",
    data_source="jaffle_shop",
    dimensions=[
        Dimension(name="id", sql="id", type="string", primary_key=True),
        Dimension(name="segment", sql="segment", type="string"),
    ],
    measures=[Measure(name="count", type="count")],
)
storage.save_model(segments_model)

print("customer_segments model saved")

In [ ]:
# Query orders with a dynamic join to customer_segments
result = engine.execute(query=SlayerQuery(
    source_model={
        "source_name": "orders",
        "joins": [{"target_model": "customer_segments", "join_pairs": [["customer_id", "id"]]}],
    },
    fields=[{"formula": "count"}, {"formula": "order_total_sum"}],
    dimensions=[{"name": "customer_segments.segment"}],
    order=[{"column": {"name": "order_total_sum"}, "direction": "desc"}],
))

print("Revenue by customer segment (dynamic join):")
for row in result.data:
    segment = row["orders.customer_segments.segment"]
    count = row["orders.count"]
    revenue = row["orders.order_total_sum"]
    print(f"  {segment}: {count:,} orders, ${revenue:,.2f}")

## Summary

SLayer's join system provides:

| Feature | Description |
|---------|-------------|
| **LEFT JOINs** | The only join type, used for data enrichment |
| **Dot syntax** | `customers.name`, `orders.customers.regions.name` in queries |
| **`__` aliases** | `orders__customers.name` in model SQL definitions |
| **Auto-ingestion** | FK constraints become `ModelJoin` objects automatically |
| **Diamond joins** | Same table via multiple paths gets separate path-based aliases |
| **Dynamic joins** | `ModelExtension` adds joins at query time without modifying models |

See the [Models docs](../../concepts/models.md#joins) and [Ingestion docs](../../concepts/ingestion.md) for the full reference.